# PyTorch Migration Notebook

Notebook ini adalah port awal dari workflow TensorFlow ke PyTorch untuk Windows native dengan GPU NVIDIA. Struktur dataset tetap memakai folder `data/processed/<split>/<class>`.

In [1]:
import os
import time
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("GPU : not available")

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

plt.style.use("seaborn-v0_8-whitegrid")

Device : cuda
GPU : NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 25
PATIENCE = 7
INITIAL_LR = 1e-4
FINE_TUNE_LR = 1e-5

TRAIN_DIR = Path("data/processed/train")
VAL_DIR = Path("data/processed/validation")
TEST_DIR = Path("data/processed/test")

NUM_CLASSES = 4
class_names = ["healthy leaf", "leaf curl", "leaf spot", "yellowish"]

REPORT_DIR = Path("reports")
FIG_DIR = REPORT_DIR / "figures"
METRICS_DIR = REPORT_DIR / "metrics"
FIG_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=eval_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

print("Train samples :", len(train_dataset))
print("Validation samples :", len(val_dataset))
print("Test samples :", len(test_dataset))
print("Class mapping :", train_dataset.class_to_idx)

Train samples : 2240
Validation samples : 240
Test samples : 240
Class mapping : {'healthy leaf': 0, 'leaf curl': 1, 'leaf spot': 2, 'yellowish': 3}


In [5]:
def build_mobilenet_v2(num_classes=NUM_CLASSES):
    model = models.mobilenet_v2(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

def build_efficientnet_b0(num_classes=NUM_CLASSES):
    model = models.efficientnet_b0(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

class FusionNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.mobilenet = models.mobilenet_v2(weights=None).features
        self.efficientnet = models.efficientnet_b0(weights=None).features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(1280 + 1280, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x1 = self.pool(self.mobilenet(x)).flatten(1)
        x2 = self.pool(self.efficientnet(x)).flatten(1)
        x = torch.cat([x1, x2], dim=1)
        return self.classifier(x)

MODEL_BUILDERS = {
    "MobileNetV2": build_mobilenet_v2,
    "EfficientNetB0": build_efficientnet_b0,
    "Fusion Model": FusionNet,
}


def run_sanity_check():
    images, labels = next(iter(train_loader))
    print("Batch images shape :", images.shape)
    print("Batch labels shape :", labels.shape)

    sample_model = build_mobilenet_v2(NUM_CLASSES).to(device)
    sample_model.eval()
    with torch.no_grad():
        outputs = sample_model(images.to(device))
    print("Sample output shape :", outputs.shape)
    print("Expected classes :", NUM_CLASSES)

run_sanity_check()

Batch images shape : torch.Size([32, 3, 224, 224])
Batch labels shape : torch.Size([32])
Sample output shape : torch.Size([32, 4])
Expected classes : 4


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        running_correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total

@torch.no_grad()
def evaluate_loss_accuracy(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        running_correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total

def fit_model(model, train_loader, val_loader, epochs=EPOCHS, lr=INITIAL_LR, patience=PATIENCE):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

    best_state = copy.deepcopy(model.state_dict())
    best_val_loss = float("inf")
    patience_counter = 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    start_time = time.perf_counter()
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val_loss, val_acc = evaluate_loss_accuracy(model, val_loader, criterion)
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch + 1:02d} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    total_time = time.perf_counter() - start_time
    model.load_state_dict(best_state)
    return model, history, total_time

@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    y_prob = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        y_true.extend(labels.numpy().tolist())
        y_pred.extend(outputs.argmax(dim=1).cpu().numpy().tolist())
        y_prob.extend(probs.cpu().numpy().tolist())

    return np.array(y_true), np.array(y_pred), np.array(y_prob)

In [ ]:
def build_and_train(model_name):
    if model_name == "Fusion Model":
        model = FusionNet(num_classes=NUM_CLASSES)
    else:
        model = MODEL_BUILDERS[model_name](NUM_CLASSES)

    model = model.to(device)
    model, history, train_time = fit_model(model, train_loader, val_loader)
    val_loss, val_acc = evaluate_loss_accuracy(model, val_loader, nn.CrossEntropyLoss())
    test_loss, test_acc = evaluate_loss_accuracy(model, test_loader, nn.CrossEntropyLoss())
    y_true, y_pred, y_prob = predict_proba(model, test_loader)

    report_dict = classification_report(y_true, y_pred, target_names=class_names, zero_division=0, output_dict=True)
    cm = confusion_matrix(y_true, y_pred)

    return {
        "model_name": model_name,
        "model": model,
        "history": history,
        "train_time": train_time,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "report_dict": report_dict,
        "confusion_matrix": cm,
        "accuracy": test_acc,
        "loss": test_loss,
    }

MODELS_TO_RUN = ["MobileNetV2", "EfficientNetB0", "Fusion Model"]
results = {}
for model_name in MODELS_TO_RUN:
    print(f"\n=== Training {model_name} ===")
    results[model_name] = build_and_train(model_name)

In [ ]:
summary_rows = []
for model_name, result in results.items():
    weighted = result["report_dict"]["weighted avg"]
    summary_rows.append({
        "Model": model_name,
        "Accuracy": result["test_acc"],
        "Precision": weighted["precision"],
        "Recall": weighted["recall"],
        "F1-Score": weighted["f1-score"],
        "Test Loss": result["test_loss"],
        "Training Time (sec)": result["train_time"],
    })

summary_df = pd.DataFrame(summary_rows).sort_values("Accuracy", ascending=False)
display(summary_df.round(4))

acc_df = summary_df[["Model", "Accuracy"]].copy()
metrics_df = summary_df[["Model", "Precision", "Recall", "F1-Score"]].copy()
loss_df = summary_df[["Model", "Test Loss"]].copy()

plt.figure(figsize=(8, 5))
bars = plt.bar(acc_df["Model"], acc_df["Accuracy"], color=["#4C78A8", "#F58518", "#54A24B"])
plt.ylim(0, 1)
plt.title("Perbandingan Akurasi Test per Model")
plt.ylabel("Accuracy")
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h + 0.01, f"{h:.3f}", ha="center")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (model_name, result) in zip(axes, results.items()):
    sns.heatmap(result["confusion_matrix"], annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"CM - {model_name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.tight_layout()
plt.show()

classes = np.arange(len(class_names))
plt.figure(figsize=(8, 6))
for model_name, result in results.items():
    y_true_bin = label_binarize(result["y_true"], classes=classes)
    fpr, tpr, _ = roc_curve(y_true_bin.ravel(), result["y_prob"].ravel())
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{model_name} (AUC={roc_auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Micro-Average)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
acc_df.to_csv(METRICS_DIR / "accuracy_per_model.csv", index=False)
metrics_df.to_csv(METRICS_DIR / "precision_recall_f1_per_model.csv", index=False)
loss_df.to_csv(METRICS_DIR / "final_loss_per_model.csv", index=False)
summary_df.to_csv(METRICS_DIR / "classification_summary_table.csv", index=False)

print("Export selesai :", METRICS_DIR)

for idx, (model_name, result) in enumerate(results.items(), start=1):
    plt.figure(figsize=(6, 5))
    sns.heatmap(result["confusion_matrix"], annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"confusion_matrix_{idx}_{model_name.lower().replace(' ', '_')}.png", dpi=300, bbox_inches="tight")
    plt.close()

print("Export selesai :", FIG_DIR)